#**Task 1**

import libraries

In [40]:
import pandas as pd
import sqlite3
import json

 Connect to database

In [41]:
conn = sqlite3.connect("download_db.db")

Load Data

In [42]:
members = pd.read_sql("SELECT * FROM members", conn)
books = pd.read_sql("SELECT * FROM books", conn)
checkouts = pd.read_sql("SELECT * FROM checkouts", conn)

Load JSON book catalog

In [43]:
with open("download_json.json", "r") as file:
    book_catalog = json.load(file)
book_catalog = pd.DataFrame(book_catalog)

Load HTML Reading Kickoff data

In [44]:
reading_kickoff = pd.read_html("download_html.html")[0]

Questions

 1) How much is each member borrowing?

In [45]:
q1 = """
SELECT
    members.member_id,
    members.first_name || ' ' || members.last_name AS member_name,
    COUNT(checkouts.book_id) AS total_books_borrowed
FROM members
LEFT JOIN checkouts
ON members.member_id = checkouts.member_id
GROUP BY members.member_id, member_name;"""

answer1 = pd.read_sql(q1, conn)
display(answer1)

,member_id,member_name,total_books_borrowed
0,1001,Salma Ibrahim,1
1,1002,Fares Saleh,2
2,1003,Bassel Hegazy,9
3,1004,Fares Wahba,0
4,1005,Youssef Halim,3
...,...,...,...
75,1076,Dina Wahba,7
76,1077,Lina Rashad,6
77,1078,Habiba Osman,0
78,1079,Rana Osman,10


2) Which books match a chosen author pattern?

In [46]:

q2 = """
SELECT *
FROM books
WHERE author LIKE 'A%';
"""

answer2 = pd.read_sql(q2, conn)
display(answer2)

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez


3) What are the most popular books?

In [47]:

q3 = """
SELECT
    books.title,
    COUNT(checkouts.book_id) AS checkout_count
FROM checkouts
JOIN books
ON checkouts.book_id = books.book_id
GROUP BY books.title
ORDER BY checkout_count DESC
LIMIT 5;
"""

answer3 = pd.read_sql(q3, conn)
display(answer3)

,title,checkout_count
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


4) Who are the most active readers?

In [48]:

q4 = """
SELECT
    members.first_name || ' ' || members.last_name AS member_name,
    COUNT(checkouts.book_id) AS books_borrowed
FROM members
JOIN checkouts
ON members.member_id = checkouts.member_id
GROUP BY member_name
ORDER BY books_borrowed DESC
LIMIT 10;
"""

answer4 = pd.read_sql(q4, conn)
display(answer4)


,member_name,books_borrowed
0,Aya Wahba,25
1,Sherif Saleh,21
2,Ziad Saleh,19
3,Nour Nabil,18
4,Mostafa Fouad,18
5,Youssef Hegazy,17
6,Ahmed Shafik,17
7,Adam Fahmy,17
8,Sara Rashad,16
9,Reem Osman,16


5) What does a neighborhood's activity look like further back in time?


In [49]:

q5 = """
SELECT
    members.neighborhood,
    checkouts.checkout_date,
    checkouts.book_id
FROM checkouts
JOIN members
ON checkouts.member_id = members.member_id
ORDER BY checkouts.checkout_date ASC;
"""

answer5 = pd.read_sql(q5, conn)
display(answer5)

,neighborhood,checkout_date,book_id
0,Maadi,2024-01-02,509
1,Shubra,2024-01-03,514
2,Zamalek,2024-01-04,521
3,Maadi,2024-01-05,507
4,Heliopolis,2024-01-09,505
...,...,...,...
386,Nasr City,2025-12-24,503
387,Maadi,2025-12-27,520
388,Maadi,2025-12-28,530
389,Nasr City,2025-12-28,531


Combine Sources

In [50]:
combined = checkouts.merge(
    members,
    on="member_id",)

combined = combined.merge(
    books,
    on="book_id", )
combined = pd.concat(
    [combined, reading_kickoff],
    ignore_index=True )

 Check missing values


In [51]:
print("Missing Values:")
print(combined.isnull().sum())







Missing Values:
checkout_id           26
member_id             26
book_id               26
checkout_date         26
return_date           91
first_name            26
last_name             26
grade                 62
neighborhood          26
membership_status     26
join_date             31
title                 26
author                26
Member ID            391
Book ID              391
Checkout Date        391
dtype: int64


Save final file


In [52]:
combined.to_csv("Library_Combined_Dataset.csv", index=False)

#**Task 2**

Load dataset

In [53]:
df = pd.read_csv("Library_Combined_Dataset.csv")
print("Dataset shape:")
print(df.shape)
print("Columns:")
print(df.columns)

Dataset shape:
(417, 16)
Columns:
Index(['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date',
       'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status',
       'join_date', 'title', 'author', 'Member ID', 'Book ID',
       'Checkout Date'],
      dtype='object')


Problem 1: Missing Values

In [54]:
print(df.isnull().sum())

checkout_id           26
member_id             26
book_id               26
checkout_date         26
return_date           91
first_name            26
last_name             26
grade                 62
neighborhood          26
membership_status     26
join_date             31
title                 26
author                26
Member ID            391
Book ID              391
Checkout Date        391
dtype: int64



 Fill missing values

In [55]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

Problem 2: Duplicates

In [56]:
print("Duplicate Rows:")
print(df.duplicated().sum())
df = df.drop_duplicates()


Duplicate Rows:
8


 Problem 3: Inconsistent Text Values


In [57]:
text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.lower()
print("Text values ")

Text values 


Problem 4: Invalid Member IDs

In [58]:
if "member_id" in df.columns:
    if "members_id" in globals():
        valid_members = members["member_id"].unique()
        invalid = df[~df["member_id"].isin(valid_members)]
        print("Invalid member IDs:")
        print(invalid)
        df = df[df["member_id"].isin(valid_members)]

Save cleaned dataset

In [59]:
df.to_csv("task2_cleaned_data.csv", index=False)

#**Task 3**

 Load cleaned dataset from Task 2

In [60]:
df = pd.read_csv("task2_cleaned_data.csv")

Data Fairness Analysis

 Compare neighborhoods by members and checkouts

In [61]:
fairness_analysis = df.groupby("neighborhood").agg(
    number_of_members=("member_id", "nunique"),
    number_of_checkouts=("book_id", "count")
)
print(fairness_analysis)


              number_of_members  number_of_checkouts
neighborhood                                        
heliopolis                   13                   84
maadi                        19                  106
nasr city                    15                  123
shubra                        5                   34
zamalek                      10                   62


 Find highest and lowest activityl

In [62]:

most_members = fairness_analysis["number_of_members"].idxmax()
least_members = fairness_analysis["number_of_members"].idxmin()

most_checkouts = fairness_analysis["number_of_checkouts"].idxmax()
least_checkouts = fairness_analysis["number_of_checkouts"].idxmin()


print("Neighborhood with most members:", most_members)
print("Neighborhood with least members:", least_members)

print("Neighborhood with most checkouts:", most_checkouts)
print("Neighborhood with least checkouts:", least_checkouts)

Neighborhood with most members: maadi
Neighborhood with least members: shubra
Neighborhood with most checkouts: nasr city
Neighborhood with least checkouts: shubra


Save fairness results

In [65]:
fairness_analysis.to_csv("fairness_reflection.csv")